# Testing with Pytest



## Part 1 — Pytest

## Why pytest

The standard library ships `unittest` (xUnit-style: class-based, `self.assertEqual(...)`). It works, but pytest dominates new projects for four reasons:

- **Plain `assert`.** No `self.assertEqual(a, b)` ceremony; just `assert a == b`. Pytest rewrites failing asserts to show you both sides.
- **Fixtures.** Pytest's fixture system is dependency injection for tests. Cleaner than xUnit's `setUp`/`tearDown` once you know it.
- **Parametrization.** One test function, many input cases, one decorator.
- **A huge plugin ecosystem.** `pytest-cov`, `pytest-asyncio`, `pytest-mock`, `pytest-xdist` (parallel), `pytest-django`, and hundreds more.

You can still run `unittest`-style classes under pytest — pytest discovers and runs both. So adoption is incremental.

Install: `pip install pytest`.

## The basic shape — `def test_foo():`

Pytest discovers any file matching `test_*.py` (or `*_test.py`) and any function or method matching `test_*`. The test passes if it doesn't raise. The simplest test is one line:

```python
def test_addition():
    assert 1 + 1 == 2
```

When an assertion fails, pytest doesn't just print "AssertionError" — it introspects the expression and shows you what each side actually was. This is why plain `assert` is enough; no `assertEqual` needed.

In [ ]:
# What a test file looks like (would live in tests/test_math.py)

# === content of tests/test_math.py ===
def test_addition():
    assert 1 + 1 == 2

def test_string_methods():
    assert "hello".upper() == "HELLO"

def test_list_contains():
    xs = [1, 2, 3]
    assert 2 in xs

def test_failure_introspection():
    actual = {"a": 1, "b": 2}
    expected = {"a": 1, "b": 3}
    assert actual == expected     # would fail; pytest would show the diff
# ====================================

# Run from the project root:
#   $ pytest
#   $ pytest tests/test_math.py          # one file
#   $ pytest tests/test_math.py::test_addition   # one test
print("(this cell is a stand-in for the test file; real tests live in a tests/ directory)")

## Discovery — where pytest looks

From the directory you run `pytest`, it walks the tree and picks up:

- Files matching `test_*.py` or `*_test.py`
- Inside those, classes named `Test*` (no `__init__`, by convention)
- Inside files and `Test*` classes, functions/methods named `test_*`

So a typical layout:

```text
   myproject/
     pyproject.toml
     src/
       myproject/
         core.py
     tests/
       test_core.py
       test_utils.py
       conftest.py             <- shared fixtures
```

You don't need an `__init__.py` in `tests/`. Pytest handles imports through its own discovery mechanism.

Common configuration goes in `pyproject.toml`:

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra --strict-markers --strict-config"
```

## `pytest.raises` — testing for exceptions

When a function is *supposed* to raise an exception, `pytest.raises` captures it as a context manager:

```python
import pytest

def parse_age(s):
    age = int(s)
    if age < 0:
        raise ValueError(f"age must be non-negative, got {age}")
    return age

def test_negative_age_rejected():
    with pytest.raises(ValueError, match="non-negative"):
        parse_age("-5")
```

`match=` accepts a regex against the exception's string. The `with` block must raise an exception of the expected type, or the test fails.

In [ ]:
import pytest

def parse_age(s):
    age = int(s)
    if age < 0:
        raise ValueError(f"age must be non-negative, got {age}")
    return age

# Simulated test (run inline since we're in a notebook)
try:
    with pytest.raises(ValueError, match="non-negative"):
        parse_age("-5")
    print("test passed: ValueError raised with matching message")
except Exception as e:
    print(f"test failed: {e}")

# A test that should fail (function didn't raise)
try:
    with pytest.raises(ValueError):
        parse_age("42")    # this won't raise
    print("test passed")
except Exception as e:
    print(f"test failed correctly: {type(e).__name__}: {e}")

## Fixtures — `@pytest.fixture`

A **fixture** is a function decorated with `@pytest.fixture` that provides test setup. Tests request fixtures by listing them as parameters; pytest looks them up by name and injects them.

The pattern:

```python
@pytest.fixture
def sample_users():
    return [
        {"name": "alice", "age": 30},
        {"name": "bob",   "age": 25},
    ]

def test_count_users(sample_users):
    assert len(sample_users) == 2

def test_first_user_name(sample_users):
    assert sample_users[0]["name"] == "alice"
```

Both tests *request* `sample_users` by parameter name; pytest runs the fixture function once per test (default scope) and passes the result in.

The win over xUnit `setUp`: you don't pay for setup you don't need, and you can compose fixtures (fixtures can request other fixtures).

In [ ]:
import pytest

@pytest.fixture
def sample_users():
    return [
        {"name": "alice", "age": 30},
        {"name": "bob",   "age": 25},
    ]

# Manual invocation in a notebook — pytest does this for you in a test run
users = sample_users.__wrapped__()    # the underlying function

def test_count_users(users):
    assert len(users) == 2

def test_first_user_name(users):
    assert users[0]["name"] == "alice"

test_count_users(users)
test_first_user_name(users)
print("both assertions passed")

## Fixture scopes — `function`, `class`, `module`, `session`

By default, a fixture runs once per test that requests it (scope=`"function"`). Sometimes setup is expensive — opening a database connection, starting a process. You don't want to pay that cost per test.

`@pytest.fixture(scope="module")` runs the fixture once per module, reused by every test in that file. `scope="session"` runs once for the entire pytest invocation.

Fixtures can also **yield** instead of returning — code before the `yield` is setup, code after is teardown. Use it for cleanup.

In [ ]:
import pytest

# Function scope (default) — runs once per test
@pytest.fixture(scope="function")
def cheap_thing():
    return [1, 2, 3]

# Module scope — runs once per file, shared across tests
@pytest.fixture(scope="module")
def expensive_connection():
    print("  (opening connection — runs once per module)")
    conn = {"id": "conn-001", "alive": True}
    yield conn
    print("  (closing connection — runs at module teardown)")
    conn["alive"] = False

# Session scope — runs once for the whole pytest run
@pytest.fixture(scope="session")
def app_config():
    return {"env": "test", "log_level": "DEBUG"}

## Built-in fixtures — `tmp_path`, `monkeypatch`, `capsys`, `caplog`

Pytest ships a handful of high-value built-in fixtures. The four you'll use most:

- **`tmp_path`** — a fresh `pathlib.Path` to a temporary directory, auto-cleaned. Use whenever your test needs to write files.
- **`monkeypatch`** — patch attributes, dict items, env vars; auto-undone at test end. The right tool for "set this env var just for this test."
- **`capsys`** — capture stdout/stderr printed during the test. Read with `capsys.readouterr()`.
- **`caplog`** — capture log records. Inspect with `caplog.records` or `caplog.text`.

In [ ]:
import pytest

# tmp_path — fresh temp directory, auto-cleaned
def test_writes_file(tmp_path):
    p = tmp_path / "data.txt"
    p.write_text("hello")
    assert p.read_text() == "hello"

# monkeypatch — change an env var, undo at test end
def test_env_override(monkeypatch):
    monkeypatch.setenv("MY_FLAG", "1")
    import os
    assert os.environ["MY_FLAG"] == "1"
    # After the test, MY_FLAG is restored to whatever it was

# capsys — capture printed output
def test_print_output(capsys):
    print("hello, world")
    captured = capsys.readouterr()
    assert "hello" in captured.out

print("(these would all pass when run under pytest)")

## Parametrization — one test, many inputs

`@pytest.mark.parametrize("name, value, ...", [list of tuples])` runs the same test function once per row, with each row's values bound to the named parameters.

```python
@pytest.mark.parametrize("input, expected", [
    ("42", 42),
    ("0", 0),
    ("-7", -7),
])
def test_parse_int(input, expected):
    assert int(input) == expected
```

Each row becomes a separate test case in pytest's output, named `test_parse_int[42-42]`, `test_parse_int[0-0]`, etc. When one fails, you see exactly which input row.

Useful for table-driven tests, edge-case sweeps, golden-file comparisons. **Reach for it whenever you find yourself writing three similar test functions in a row.**

In [ ]:
import pytest

@pytest.mark.parametrize("input, expected", [
    ("42", 42),
    ("0", 0),
    ("-7", -7),
    ("1_000_000", 1_000_000),
])
def test_parse_int(input, expected):
    assert int(input) == expected

# In a notebook we can't easily run pytest itself; verify the cases by hand
for input, expected in [("42", 42), ("0", 0), ("-7", -7), ("1_000_000", 1_000_000)]:
    assert int(input) == expected
print("all 4 cases passed")

# Multiple parametrize stacks for a cartesian product
@pytest.mark.parametrize("x", [1, 2, 3])
@pytest.mark.parametrize("y", ["a", "b"])
def test_combinations(x, y):
    # Runs 3 * 2 = 6 cases
    assert (x, y) in [(1, "a"), (1, "b"), (2, "a"), (2, "b"), (3, "a"), (3, "b")]

## Markers — skip, xfail, custom

A **marker** is a label on a test that influences its behaviour or selection.

Built-in:

- **`@pytest.mark.skip(reason="...")`** — unconditionally skip.
- **`@pytest.mark.skipif(condition, reason="...")`** — skip if the condition is true (e.g. `sys.version_info < (3, 11)`).
- **`@pytest.mark.xfail(reason="...")`** — *expected to fail*. If it fails, it's reported as `xfail` (not a failure). If it unexpectedly passes, it's `xpass` (warning).

Custom markers — declare in `pyproject.toml`, then label tests with them. Select at the command line.

```python
@pytest.mark.slow
def test_full_integration():
    ...

# Run only slow tests:    pytest -m slow
# Run NON-slow tests:     pytest -m "not slow"
```

In `pyproject.toml`:

```toml
[tool.pytest.ini_options]
markers = [
    "slow: marks tests as slow (deselect with '-m \"not slow\"')",
    "integration: requires external services",
]
```

The `--strict-markers` flag (recommended) makes pytest reject unknown marker names — catches typos.

## `conftest.py` — sharing fixtures

A `conftest.py` file is a special pytest file: any fixtures defined in it are visible to every test file in the same directory and below, with **no imports needed**.

Use it for fixtures that span many test files — a database connection, a `client` for testing a Flask/FastAPI app, a base config dict.

```text
   tests/
     conftest.py            <- fixtures for everything in tests/
     api/
       conftest.py          <- fixtures for api/ only (in addition to the above)
       test_users.py
       test_orders.py
     db/
       test_migrations.py
```

`conftest.py` runs implicitly — pytest discovers and loads it before running tests. Never `import` from it directly.

## Mocking — `unittest.mock` and `monkeypatch`

When your code calls into something external — a database, an API, a filesystem — you usually want to *replace* that dependency in tests with a controlled fake. Two tools:

- **`unittest.mock.patch`** — the standard-library mocking framework. Patches an attribute on a module for the duration of a test.
- **`pytest`'s `monkeypatch`** — fixture form, scoped to one test. Auto-undone.
- **`pytest-mock`** — a thin wrapper around `unittest.mock` that exposes a `mocker` fixture. Saves you the `with patch(...)` ceremony.

The pattern: replace `requests.get` with a stub that returns canned data, then assert that your code calls it with the right arguments and processes the result correctly.

In [ ]:
from unittest.mock import patch, MagicMock

# Pretend we have a function that calls requests.get
def get_user(user_id):
    import requests
    response = requests.get(f"https://api.example.com/users/{user_id}")
    return response.json()

# Test with `patch` as a context manager
with patch("requests.get") as mock_get:
    mock_get.return_value = MagicMock(json=lambda: {"id": 42, "name": "ganesh"})
    result = get_user(42)
    assert result == {"id": 42, "name": "ganesh"}
    mock_get.assert_called_once_with("https://api.example.com/users/42")

print("test passed")

## Test coverage — `pytest-cov`

Coverage tells you which lines of your code were exercised by your tests. The standard tool is `coverage.py`, wrapped for pytest by `pytest-cov`.

```bash
pip install pytest-cov
pytest --cov=src/myproject --cov-report=term-missing
```

Output shows total coverage percentage and lists uncovered lines per file. Typical configuration in `pyproject.toml`:

```toml
[tool.coverage.run]
source = ["src/myproject"]
branch = true            # also measure branch coverage

[tool.coverage.report]
fail_under = 80           # fail the CI run if coverage drops below 80%
exclude_lines = [
    "pragma: no cover",
    "raise NotImplementedError",
]
```

**100% coverage is not the goal.** 100% coverage with bad tests is worse than 80% with good ones. Aim for "every important behaviour has a test that would fail if you broke it." Coverage is a signal that *you haven't even tried* certain branches — useful as a floor, not as a target.

## CI integration — what to run, what to gate on

A reasonable CI pipeline for a Python project:

```yaml
- pip install -e .[dev]
- ruff check src tests
- ruff format --check src tests
- mypy src
- pytest -ra --cov=src --cov-report=term --cov-fail-under=80
```

Gate the merge on **all of them**. The cost of failing a check at PR time is small. The cost of a regression in main is large.

Run pytest with `-ra` to show short summaries of failures and skips at the end. Add `-x` (stop on first failure) for local iteration. Use `pytest-xdist` (`pytest -n auto`) to parallelize across cores once the suite is large enough to matter.

## Part 2 — Standard library essentials

A consolidated map of the modules you'll reach for most. Many you've already seen across this track — included here as pointers so you remember what's available without reaching for `pip install`.

## `datetime` — dates, times, durations, time zones

```python
from datetime import date, time, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

date.today()                          # date(2026, 6, 3)
datetime.now(timezone.utc)            # aware UTC timestamp
datetime.now(ZoneInfo("Asia/Kolkata"))  # aware local timestamp

date(2026, 1, 1) + timedelta(days=90) # add a duration

# Parsing — ISO format is best
datetime.fromisoformat("2026-06-03T14:30:00+05:30")

# Formatting — strftime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")
```

**Always use timezone-aware datetimes for any timestamp that matters.** Naive datetimes (no `tzinfo`) are a footgun — they silently assume "local time" in ways that break across deployments.

`zoneinfo` (3.9+) reads the system's IANA tzdata. On Windows, install `tzdata` from PyPI.

## `pathlib` — paths

Deep-dived in notebook 06. The recap:

- `Path("data") / "users.txt"` — compose with `/`
- `.read_text()`, `.write_text()`, `.read_bytes()`, `.write_bytes()`
- `.exists()`, `.is_file()`, `.is_dir()`, `.mkdir(parents=True, exist_ok=True)`
- `.glob("*.csv")`, `.rglob("**/*.py")`
- `.resolve()`, `.relative_to(other)`, `.parent`, `.stem`, `.suffix`

Never use `os.path.join`. Always use `pathlib.Path`.

## `json` — read and write JSON

```python
import json

# String <-> Python
data = json.loads('{"name": "ganesh", "age": 30}')
text = json.dumps(data, indent=2, sort_keys=True)

# File <-> Python
with open("data.json", encoding="utf-8") as f:
    data = json.load(f)

with open("out.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)
```

`json` handles dicts, lists, strings, numbers, bools, None. For datetimes, custom classes, etc., you need to provide a `default=` function (or use a typed-JSON library like `pydantic` or `orjson` + dataclass-codecs).

For performance-sensitive code: **`orjson`** (third-party) is several times faster than `json` and handles datetimes/UUIDs out of the box.

## `re` — regular expressions

```python
import re

# Match
m = re.search(r"\d{3}-\d{4}", "phone: 555-1212")
if m:
    print(m.group())   # 555-1212

# Compile for repeated use
pattern = re.compile(r"^(\w+)@(\w+\.\w+)$")
m = pattern.match("ganesh@example.com")
print(m.groups())     # ('ganesh', 'example.com')

# Substitution
re.sub(r"\s+", " ", "many   spaces    here")   # 'many spaces here'

# All matches
re.findall(r"\b\w{5}\b", "this is hello world from python")   # 5-letter words
```

Use raw strings (`r"..."`) for patterns — saves the backslash-escaping dance.

For complex parsing — JSON, CSV, structured data — reach for the dedicated library before reaching for regex. Regex is the right tool for "find these patterns in unstructured text," not for parsing grammars.

## `argparse` — command-line arguments

```python
import argparse

parser = argparse.ArgumentParser(description="Process some data.")
parser.add_argument("input", help="input file path")
parser.add_argument("-o", "--output", default="out.txt", help="output file path")
parser.add_argument("-v", "--verbose", action="store_true", help="enable verbose mode")
parser.add_argument("--retries", type=int, default=3, help="number of retries")

args = parser.parse_args()
print(args.input, args.output, args.verbose, args.retries)
```

`argparse` is in the standard library and is enough for most CLIs. For richer interfaces (subcommands, nicer help, type hints), the most-reached-for third-party tools are **`click`** and **`typer`** (typer is built on click and uses type hints to define the interface).

## Standard library — the rest worth knowing

Brief pointers; each shows up in earlier notebooks or is straightforward.

| Module | What for |
|---|---|
| `logging` | structured logging — covered in notebook 08 |
| `collections` | `defaultdict`, `Counter`, `namedtuple`, `OrderedDict` — notebook 04 |
| `itertools` | iterator combinators — notebook 07 |
| `functools` | `cache`, `lru_cache`, `partial`, `reduce`, `singledispatch` — notebooks 07, 08 |
| `os` | environment variables, process info; for paths use `pathlib` instead |
| `sys` | `sys.argv`, `sys.exit`, `sys.path`, `sys.version_info` |
| `subprocess` | run external commands; `subprocess.run(["ls", "-la"], capture_output=True)` |
| `urllib` | basic HTTP; prefer `requests` for real work |
| `csv` | read/write CSV files |
| `sqlite3` | a real SQL database in the standard library |
| `tomllib` | read TOML (3.11+); for writing use third-party `tomli-w` |
| `dataclasses` | typed records — notebook 05 |
| `enum` | enumerations |
| `hashlib` | cryptographic hashes (SHA256, etc.) |
| `secrets` | crypto-safe random — for tokens, passwords |
| `uuid` | UUID generation |
| `decimal` | exact decimal arithmetic — notebook 02 |
| `statistics` | mean, median, stdev — for quick stats without NumPy |

## The end of the track

That's the language. You now have:

- A working Python toolchain (notebook 01)
- The value, operator, and control-flow semantics (notebooks 02, 03)
- The container types and the comprehension shorthand (notebook 04)
- Object-oriented Python with classes and dataclasses (notebook 05)
- Errors, files, modules, and the import system (notebook 06)
- Iteration, generators, decorators (notebook 07)
- Type hints and the style automation that keeps code legible (notebook 08)
- Concurrency — threads, processes, asyncio — and when to reach for each (notebook 09)
- Testing with pytest, plus the standard-library map (this notebook)

What comes next is applied — the libraries and frameworks that build on this foundation. Data work with `pandas` and `numpy`. Web services with `fastapi`. Machine learning with `scikit-learn` and `pytorch`. Data engineering with Apache Spark (the `apache-spark/` repo bridges directly — PySpark uses the same Python you've learned here).

The shapes from this track — `dataclass` records, `Optional` / `None`-aware code, `for x in xs` over iterables, decorators around hot paths, fixtures around tests — show up in every Python codebase you'll touch.